# Part 4 — GenAI: turn call-centre notes into structured reasons
The ML model says WHO is at risk; the call-centre notes say WHY patients missed.
Gemini reads each note and returns a fixed JSON format, which Pydantic then validates.

In [1]:
!pip install -q google-genai pydantic

In [ ]:
import json
import time
import pandas as pd
from google import genai
from google.genai import types

API_KEY = input("Paste your key: ")      # typed at run time — never written in the code
client = genai.Client(api_key=API_KEY)
MODEL_NAME = "gemini-3.5-flash"          # the Flash model from Session 22.1
print("Client ready")

## Task 4.1 — One validated call
The system instruction (how to behave) stays general. The allowed root_cause values go in the CRAFT prompt,
so that in Task 4.3 the only difference between prompt V1 and V2 is the prompt itself.

In [3]:
ALLOWED = ["forgot_or_no_reminder", "distance_or_transport", "cost_or_fee",
           "illness_or_emergency", "work_or_schedule", "other"]

# System instruction = HOW the model should behave (same for every call)
SYSTEM_TEXT = ("You are a healthcare analytics assistant for CareLoop Clinics. "
               "Read call-centre notes carefully, do not invent facts, and reply only with JSON.")

# JSON schema = the exact shape of the reply (same idea as the name/age example in class)
SCHEMA = {
    "type": "object",
    "properties": {"root_cause": {"type": "string"}, "confidence": {"type": "number"},
                   "summary": {"type": "string"}, "recommended_action": {"type": "string"}},
    "required": ["root_cause", "confidence", "summary", "recommended_action"],
}

def build_prompt(note_text):
    """CRAFT prompt: Context, Role, Action, Tone, Format."""
    return f"""Context: A patient missed an appointment at an outpatient clinic. The call centre phoned the patient afterwards and wrote this note:
"{note_text}"

Role: You are a healthcare analytics assistant.

Action: Find the main reason the patient missed the appointment. Set root_cause to exactly one of these values:
{", ".join(ALLOWED)}
If the note does not clearly show a reason, answer "other" rather than guess.

Tone: Professional and empathetic.

Format: Return JSON with:
- root_cause: one value from the list above
- confidence: a number from 0 to 1 (how sure you are)
- summary: at most 25 words
- recommended_action: one specific follow-up step for the front desk
"""

In [5]:
API_CALLS = 0      # counts every real call to Gemini (Task 4.3 asks for the total)

def call_gemini(prompt):
    """Send one prompt to Gemini. Returns the reply text exactly as it came back."""
    global API_CALLS                # lets this function change the API_CALLS variable outside it
    API_CALLS += 1
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,                          # same input -> same answer
            max_output_tokens=1000,                 # plenty of room so the JSON is not cut off
            response_mime_type="application/json",  # reply in JSON
            response_schema=SCHEMA,                 # with exactly these 4 fields
            thinking_config=types.ThinkingConfig(thinking_level="minimal"),   # as in Session 22.1
        ),
    )
    return response.text

def analyse_note(note_text):
    """Task 4.1: one note in -> parsed Python dictionary out."""
    raw = call_gemini(build_prompt(note_text))
    return json.loads(raw)          # JSON text -> dictionary

In [6]:
notes = pd.read_csv("data/followup_notes.csv")

# Pick the text of note N001
note_n001 = notes.loc[notes["note_id"] == "N001", "note_text"].iloc[0]
print("Note:", note_n001)

result = analyse_note(note_n001)     # 1 API call
print("\nParsed reply:")
print(result)
print("Fields:", list(result.keys()))

Note: Patient said she never got any reminder message and completely forgot about the visit. Wants to rebook for next week.

Parsed reply:
{'root_cause': 'forgot_or_no_reminder', 'confidence': 0.98, 'summary': 'Patient missed her appointment because she received no reminder message and forgot. She requested to reschedule for next week.', 'recommended_action': "Reschedule the appointment for next week and verify the patient's contact details to ensure SMS reminders are enabled."}
Fields: ['root_cause', 'confidence', 'summary', 'recommended_action']


**Result:** analyse_note returned a dictionary with exactly the 4 fields. For N001, root_cause = forgot_or_no_reminder
(confidence 0.98).

## Task 4.2 — Validate and process the 10 notes
(a) The rules every Gemini reply must pass: a Pydantic model, plus a check that root_cause is an allowed value.

In [7]:
from pydantic import BaseModel, Field, ValidationError

class FollowupInsight(BaseModel):
    """The rules every Gemini reply must pass."""
    root_cause: str = Field(min_length=3)            # text, at least 3 characters
    confidence: float = Field(ge=0, le=1)            # number from 0 to 1 (ge = >=, le = <=)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

def is_allowed(root_cause):
    """Extra check: root_cause must be one of the 6 allowed values."""
    return root_cause in ALLOWED

# Quick test on the N001 reply from Task 4.1 (no API call)
insight = FollowupInsight(**result)      # ** unpacks the dictionary into the 4 fields
print("Pydantic check passed:", insight)
print("root_cause allowed:", is_allowed(insight.root_cause))

Pydantic check passed: root_cause='forgot_or_no_reminder' confidence=0.98 summary='Patient missed her appointment because she received no reminder message and forgot. She requested to reschedule for next week.' recommended_action="Reschedule the appointment for next week and verify the patient's contact details to ensure SMS reminders are enabled."
root_cause allowed: True


In [8]:
# (b) Only the first 10 notes (free-tier limit)
first10 = notes.head(10)

for _, note in first10.iterrows():
    text = note["note_text"]
    # Blank = missing (NaN) OR only spaces. pd.isna() must come first: NaN has no .strip()
    is_blank = pd.isna(text) or str(text).strip() == ""
    print(note["note_id"], "| blank" if is_blank else "| ok   ", "|", text)

N001 | ok    | Patient said she never got any reminder message and completely forgot about the visit. Wants to rebook for next week.
N002 | ok    | Lives far from the clinic and could not find any transport that morning. Says the clinic is too far for a routine check-up.
N003 | ok    | Patient felt the consultation fee was too high this month and decided to skip. Asked if there is any discount for follow-ups.
N004 | blank | nan
N005 | ok    | Had high fever and body ache on the day, stayed in bed. Will visit once recovered.
N006 | ok    | Manager called an urgent meeting at the office, could not leave work. Asked for an evening slot next time.
N007 | ok    | Bhool gaya, SMS nahi aaya tha. Dobara booking karni hai.
N008 | ok    | City bus strike that day, no auto available near the colony. Distance is a big problem for him.
N009 | ok    | Phone was switched off, could not reach the patient after three attempts.
N010 | ok    | Had a shift change at the factory and could not take leave. P

In [9]:
rows = []

for _, note in first10.iterrows():
    text = note["note_text"]

    # One result row per note; all fields start empty
    row = {"note_id": note["note_id"], "appointment_id": note["appointment_id"],
           "root_cause": None, "confidence": None, "summary": None,
           "recommended_action": None, "validation_status": None, "raw_response": None}

    # Input validation: blank note -> fixed values, NO API call
    if pd.isna(text) or str(text).strip() == "":
        row["root_cause"] = "other"
        row["confidence"] = 0.0
        row["summary"] = "No note recorded"
        row["recommended_action"] = "Re-call patient"
        row["validation_status"] = "skipped_blank"
        row["raw_response"] = ""
        rows.append(row)
        print(f"{note['note_id']}: skipped_blank (no API call)")
        continue                                   # jump to the next note

    try:
        raw = call_gemini(build_prompt(text))      # 1 API call
        row["raw_response"] = raw                  # keep the reply exactly as returned
        data = json.loads(raw)                     # JSON text -> dictionary (fails if the JSON is broken)

        # Copy whichever fields the reply has; a missing field stays empty (never repaired)
        for field in ["root_cause", "confidence", "summary", "recommended_action"]:
            row[field] = data.get(field)

        FollowupInsight(**data)                    # raises ValidationError if a rule fails
        if is_allowed(data["root_cause"]):
            row["validation_status"] = "valid"
        else:
            row["validation_status"] = "invalid"   # passed Pydantic, but not one of the 6 allowed values

    except Exception as e:                         # broken JSON, failed rule or API error -> record it, carry on
        row["validation_status"] = "invalid"
        print(f"   problem with {note['note_id']}: {type(e).__name__}")

    rows.append(row)
    print(f"{note['note_id']}: {row['root_cause']} ({row['validation_status']})")
    time.sleep(13)                                 # wait 13 seconds: free-tier rate limit

print("\nAPI calls so far:", API_CALLS)

N001: forgot_or_no_reminder (valid)
N002: distance_or_transport (valid)
N003: cost_or_fee (valid)
N004: skipped_blank (no API call)
N005: illness_or_emergency (valid)
N006: work_or_schedule (valid)
N007: forgot_or_no_reminder (valid)
   problem with N008: ServerError
N008: None (invalid)
N009: other (valid)
N010: work_or_schedule (valid)

API calls so far: 10


In [10]:
# N008 got a ServerError: Gemini's server failed, so there was NO reply to keep.
# Retrying the call once is not "repairing" a reply. Same steps as in Cell 12, for this one note.
i = 7                          # N008 is the 8th note -> position 7 (counting starts at 0)
note = first10.iloc[i]
row = rows[i]                  # the SAME dictionary that sits in rows, so changes here update rows

try:
    raw = call_gemini(build_prompt(note["note_text"]))    # 1 API call
    row["raw_response"] = raw
    data = json.loads(raw)
    for field in ["root_cause", "confidence", "summary", "recommended_action"]:
        row[field] = data.get(field)
    FollowupInsight(**data)
    if is_allowed(data["root_cause"]):
        row["validation_status"] = "valid"
    else:
        row["validation_status"] = "invalid"
except Exception as e:
    row["validation_status"] = "invalid"
    print("Still a problem:", type(e).__name__)

print(f"{row['note_id']}: {row['root_cause']} ({row['validation_status']})")
print("API calls so far:", API_CALLS)

N008: distance_or_transport (valid)
API calls so far: 11


In [11]:
# (d) Save every row (valid, invalid and skipped) in the column order the brief asks for
analysed = pd.DataFrame(rows, columns=["note_id", "appointment_id", "root_cause", "confidence",
                                       "summary", "recommended_action", "validation_status",
                                       "raw_response"])
analysed.to_csv("notes_analysed.csv", index=False)

print("Shape:", analysed.shape)
print(analysed[["note_id", "root_cause", "confidence", "validation_status"]])
print("\nStatus counts:")
print(analysed["validation_status"].value_counts())

Shape: (10, 8)
  note_id             root_cause  confidence validation_status
0    N001  forgot_or_no_reminder        0.98             valid
1    N002  distance_or_transport        0.95             valid
2    N003            cost_or_fee        0.95             valid
3    N004                  other        0.00     skipped_blank
4    N005   illness_or_emergency        0.98             valid
5    N006       work_or_schedule        0.98             valid
6    N007  forgot_or_no_reminder        0.95             valid
7    N008  distance_or_transport        0.98             valid
8    N009                  other        0.90             valid
9    N010       work_or_schedule        0.95             valid

Status counts:
validation_status
valid            9
skipped_blank    1
Name: count, dtype: int64


**Result:** notes_analysed.csv has 10 rows and 8 columns. N004 was blank → skipped_blank, no API call.
The other 9 notes are all valid, and all 9 root causes match the expected answers.
N008 first failed with a ServerError (Gemini's server, no reply returned); that one call was retried once and then succeeded.
API calls so far: 11 (1 in Task 4.1 + 9 in the loop + 1 retry).

## Task 4.3 — Break it on purpose
(a) Feed Pydantic two deliberately bad replies and check that each one is caught.

In [13]:
# Bad reply 1: confidence is above 1
bad_1 = {"root_cause": "cost_or_fee", "confidence": 1.7,
         "summary": "Fee was too high this month.", "recommended_action": "Offer a follow-up discount."}
try:
    FollowupInsight(**bad_1)
    print("Bad reply 1 passed - this should not happen")
except ValidationError as e:
    print("Bad reply 1 caught:")
    print(e)

# Bad reply 2: recommended_action is missing
bad_2 = {"root_cause": "cost_or_fee", "confidence": 0.8,
         "summary": "Fee was too high this month."}
try:
    FollowupInsight(**bad_2)
    print("Bad reply 2 passed - this should not happen")
except ValidationError as e:
    print("\nBad reply 2 caught:")
    print(e)

Bad reply 1 caught:
1 validation error for FollowupInsight
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal

Bad reply 2 caught:
1 validation error for FollowupInsight
recommended_action
  Field required [type=missing, input_value={'root_cause': 'cost_or_f...s too high this month.'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing


**Result:** both bad replies were caught without crashing.
- confidence = 1.7 → "Input should be less than or equal to 1" (the `le=1` rule)
- missing recommended_action → "Field required" (every field without a default is required)

(b) Prompt comparison on N007 (Hinglish) and N009 (ambiguous). V1 = "Classify this note" only;
V2 = the CRAFT prompt with the allowed values. Same schema and settings — only the prompt differs.

In [14]:
def build_prompt_v1(note_text):
    """V1: the lazy prompt — no context, no allowed values."""
    return f'Classify this note: "{note_text}"'

tests = []
for note_id in ["N007", "N009"]:
    text = notes.loc[notes["note_id"] == note_id, "note_text"].iloc[0]

    for version in ["V1", "V2"]:
        if version == "V1":
            prompt = build_prompt_v1(text)
        else:
            prompt = build_prompt(text)              # the CRAFT prompt from Task 4.1

        try:
            raw = call_gemini(prompt)                # 1 API call
            root_cause = json.loads(raw).get("root_cause")
        except Exception as e:
            root_cause = None
            print(f"   problem with {note_id} {version}: {type(e).__name__}")

        tests.append({"note": note_id, "prompt": version,
                      "root_cause": root_cause, "allowed": is_allowed(root_cause)})
        time.sleep(13)                               # free-tier rate limit

comparison = pd.DataFrame(tests)
print(comparison)
print("\nAPI calls so far:", API_CALLS)

   note prompt                                    root_cause  allowed
0  N007     V1  Unreceived SMS reminder / Missed appointment    False
1  N007     V2                         forgot_or_no_reminder     True
2  N009     V1                           Unreachable patient    False
3  N009     V2                                         other     True

API calls so far: 15


**Result:**

| note | prompt | root_cause | allowed |
|---|---|---|---|
| N007 (Hinglish) | V1 | Unreceived SMS reminder / Missed appointment | False |
| N007 (Hinglish) | V2 | forgot_or_no_reminder | True |
| N009 (ambiguous) | V1 | Unreachable patient | False |
| N009 (ambiguous) | V2 | other | True |

**Deploy V2.** With V1, Gemini invents its own labels, so the replies can't be counted or grouped across notes.
V2 lists the allowed values and tells the model to answer "other" rather than guess, so both the Hinglish note and the
ambiguous note get a usable, allowed root_cause.

In [15]:
# (c) Total API calls across Tasks 4.1-4.3 (counted inside call_gemini)
print("Total API calls used:", API_CALLS)
print("Breakdown: 4.1 = 1 | 4.2 = 9 + 1 retry (N008) | 4.3 = 4  ->", 1 + 9 + 1 + 4)

Total API calls used: 15
Breakdown: 4.1 = 1 | 4.2 = 9 + 1 retry (N008) | 4.3 = 4  -> 15


**(c)** Total API calls: 15 (4.1: 1, 4.2: 9 plus 1 retry after N008's ServerError, 4.3: 4) — below the 25-call budget.
The brief's estimate is 12–14; the one extra call is the N008 retry.

## Task 4.4 — Combine the AI reasons with the booking data
Join each analysed note to its appointment, then check how the reasons relate to whether a reminder was sent.

In [16]:
notes_out = pd.read_csv("notes_analysed.csv")
appts = pd.read_csv("appointments_clean.csv")

# Join: each note gets the booking details of its appointment
merged = notes_out.merge(appts, on="appointment_id", how="left")
print("Merged rows:", len(merged))
print(merged[["note_id", "appointment_id", "root_cause", "reminder_sent", "status"]])

# Count of each reason, split by whether a reminder was sent (margins=True adds the totals)
print("\nroot_cause vs reminder_sent:")
print(pd.crosstab(merged["root_cause"], merged["reminder_sent"], margins=True))

Merged rows: 10
  note_id appointment_id             root_cause reminder_sent   status
0    N001          A0872  forgot_or_no_reminder            No  No-Show
1    N002          A0897  distance_or_transport            No  No-Show
2    N003          A0526            cost_or_fee           Yes  No-Show
3    N004          A0926                  other            No  No-Show
4    N005          A0534   illness_or_emergency           Yes  No-Show
5    N006          A0507       work_or_schedule           Yes  No-Show
6    N007          A0333  forgot_or_no_reminder            No  No-Show
7    N008          A0222  distance_or_transport           Yes  No-Show
8    N009          A0977                  other           Yes  No-Show
9    N010          A0596       work_or_schedule            No  No-Show

root_cause vs reminder_sent:
reminder_sent          No  Yes  All
root_cause                         
cost_or_fee             0    1    1
distance_or_transport   1    1    2
forgot_or_no_reminder   2    

**Result:** the merged table has 10 rows. Both notes classed forgot_or_no_reminder (N001, N007) belong to appointments
where reminder_sent = No (2 of 2), and 5 of the 10 notes belong to appointments with no reminder.
This matches Part 3, where reminder_sent had the most negative coefficient. With only 10 notes, this is an illustration,
not proof of the clinic-wide pattern.